In [140]:
import pandas as pd
import numpy as np
import geopandas as gpd

In [141]:
# Load data
jobs = pd.read_csv("../data/irec_solar_jobs.csv")
rockefeller = pd.read_csv("../data/rockefeller_eig_index.csv", dtype={"geoid10": str})
results = pd.read_csv("../data/dollar_per_watt_agent_results.csv", dtype={"county_id": str})
map = pd.read_csv("../data/dgen_county_fips_mapping.csv", dtype={"geoid10": str, "county_id": str})
states = gpd.read_file("../data/states.shp")

# RESIDENTIAL PROPORTION
RES_PROP = (100814+30696)/178812 # From IREC report

In [142]:
# Increase jobs to 2024 levels based on projected growth
jobs['installation_proj_dev_jobs_2024'] = (
    jobs['installation_proj_dev_jobs_2023'] * (1 + jobs['projected_growth_2024'])
)


In [143]:
# Join county fips codes to dgen results
results_geoid = results.merge(map[['geoid10', 'county_id']], on='county_id', how = 'left')

# Calculate proportion of state HH each geoid represents
results_geoid['hh'] = results_geoid['customers_in_bin']/results_geoid['pct_of_bldgs_developable']
results_geoid['state_hh'] = results_geoid.groupby(['state_abbr', 'scenario'])['hh'].transform('sum')
results_geoid['county_prop'] = results_geoid['hh'] / results_geoid['state_hh']
results_geoid_agg = (
    results_geoid
    .groupby(['geoid10'], as_index = False)
    .agg({'county_prop':'sum', 'state_hh':'mean'})
)

# Aggregate 2040 cumulative adoption by geoid, scenario, then pivot out
results_2040 = (
    results_geoid[results_geoid['year'] == 2040]
    .groupby(['state_abbr','geoid10', 'scenario'], as_index = False)
    .agg({'number_of_adopters': 'sum'})
    .pivot(index = ['geoid10', 'state_abbr'], columns = 'scenario', values = 'number_of_adopters')
    .reset_index()
)


# Calculate percentage difference between scenarios
results_2040['pct_diff'] = (
    results_2040['policy'] / results_2040['baseline']
)

# Join back to state hh proportions
results_2040 = results_2040.merge(
    results_geoid_agg[['geoid10', 'county_prop', 'state_hh']].drop_duplicates(),
    on = ['geoid10'],
    how = 'left'
)

# Join to jobs
results_jobs = (
    results_2040
    .merge(jobs[['state_abbr', 'installation_proj_dev_jobs_2024', 'five_year_growth']], on='state_abbr', how='left')
)

# Distribute jobs to counties based on proportion of state hh
results_jobs['county_jobs'] = (
    results_jobs['installation_proj_dev_jobs_2024'] * RES_PROP * results_jobs['county_prop']
)

# Calculate baseline 2040 jobs
results_jobs['county_jobs_baseline'] = np.where(
    results_jobs['five_year_growth'] > 0,
    results_jobs['county_jobs'] * (1 + (results_jobs['five_year_growth']/5)**15),
    results_jobs['county_jobs'] * (1.01**15)
)

# Calculate job increase based on pct diff in adoption
results_jobs['dollar_per_watt_jobs'] = (
    results_jobs['county_jobs_baseline'] * results_jobs['pct_diff']
)

# Join to rockefeller data
results_rockefeller = (
    results_jobs
    .merge(rockefeller[['geoid10', 'quintile']], on='geoid10', how='left')
)

# Calculate difference
results_rockefeller['job_increase'] = (
    results_rockefeller['dollar_per_watt_jobs'] - results_rockefeller['county_jobs_baseline']
)

In [144]:
results = (
    results_rockefeller
    .groupby(['state_abbr', 'quintile'], as_index = False).agg({'county_jobs_baseline':'sum','dollar_per_watt_jobs': 'sum', 'job_increase':'sum', 'state_hh':'mean'})
)

results['jobs_per_100k'] = (
    results_rockefeller['job_increase'] / (results_rockefeller['state_hh']/100000)
)

In [155]:
results['dollar_per_watt_jobs'].sum()

np.float64(739044.3620712645)

In [156]:
results['county_jobs_baseline'].sum()

np.float64(282545.16189725325)

In [145]:
pivot = (
    results
    .pivot(index = 'state_abbr', columns = 'quintile', values = 'job_increase')
    .reset_index()
)

In [150]:
pivot.columns = ['state_abbr', 'prosperous', 'comfortable', 'mid_tier', 'at_risk', 'distressed']

In [153]:
# Exclude the state column and coerce to numeric (non-numeric -> NaN)
num = pivot.drop(columns=['state_abbr']).apply(pd.to_numeric, errors='coerce')

# Column-wise sums (NaNs ignored by default)
col_sums = num.sum(skipna=True)

print("Column-wise sums (excluding 'state_abbr' and ignoring NAs):")
for col, val in col_sums.items():
    print(f"- {col}: {round(val)}")

Column-wise sums (excluding 'state_abbr' and ignoring NAs):
- prosperous: 190170
- comfortable: 121061
- mid_tier: 87629
- at_risk: 35405
- distressed: 22234


In [157]:
190170 + 121061 + 87629

398860

In [146]:
pivot.to_csv("../outputs/rockefeller_jobs_by_state.csv", index = False)